# ML4 — Supervised Learning. Classification
## Loyiha yechimi: "Don't Get Kicked" (Kaggle)

Bu notebook loyihaning V bobidagi barcha 12 ta vazifani bosqichma-bosqich yechadi. Har bir bo'lim vazifa matnini, tushuntirishni va kodni o'z ichiga oladi.

**Eslatma:** Ma'lumotlar [Kaggle "Don't Get Kicked"](https://www.kaggle.com/c/DontGetKicked) sahifasidan yuklab olinib, `data/training.csv` sifatida saqlangan deb faraz qilinadi (README'dagi ko'rsatmaga muvofiq). Agar fayl nomi boshqacha bo'lsa, quyidagi `DATA_PATH` o'zgaruvchisini moslang.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, average_precision_score

RANDOM_STATE = 42
DATA_PATH = "data/training.csv"   # Kaggle'dan yuklangan fayl shu yerda bo'lishi kutiladi
TARGET = "IsBadBuy"
DATE_COL = "PurchDate"

pd.set_option("display.max_columns", 100)

## 1-vazifa. Ma'lumotlarni yuklab olish

Ma'lumotlar [Kaggle "Don't Get Kicked"](https://www.kaggle.com/c/DontGetKicked) musobaqasidan yuklab olinadi. Kaggle API o'rnatilgan bo'lsa:

```bash
kaggle competitions download -c DontGetKicked -p data/
unzip data/DontGetKicked.zip -d data/
```

Natijada `data/training.csv` va `data/test.csv` fayllari hosil bo'ladi. Biz `training.csv`dan train/valid/test bo'linishini o'zimiz quramiz (2-vazifa), chunki Kaggle'ning o'z `test.csv`'sida `IsBadBuy` yorlig'i yo'q (bu — musobaqa uchun yashirin baholash fayli).

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Target taqsimoti
print(df[TARGET].value_counts(normalize=True))

## 2-vazifa. Train / Validation / Test bo'linishi (vaqt bo'yicha)

`PurchDate` bo'yicha saralab, sanalarning birinchi 1/3 qismini **train**, o'rtadagi 1/3 qismini **validation**, oxirgi 1/3 qismini **test** sifatida ajratamiz. Shart: `train.PurchDate < valid.PurchDate < test.PurchDate`.

In [ ]:
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

n = len(df)
i1 = n // 3
i2 = 2 * n // 3

train_df = df.iloc[:i1].copy()
valid_df = df.iloc[i1:i2].copy()
test_df  = df.iloc[i2:].copy()

print("Train:", train_df.shape, train_df[DATE_COL].min(), "->", train_df[DATE_COL].max())
print("Valid:", valid_df.shape, valid_df[DATE_COL].min(), "->", valid_df[DATE_COL].max())
print("Test :", test_df.shape,  test_df[DATE_COL].min(),  "->", test_df[DATE_COL].max())

assert train_df[DATE_COL].max() <= valid_df[DATE_COL].min()
assert valid_df[DATE_COL].max() <= test_df[DATE_COL].min()

## 3-vazifa. Kategorial belgilarni kodlash

Har bir kategorial ustun uchun `LabelEncoder`ni **faqat train**ga moslashtiramiz (`fit`), so'ng validation va test'ga `transform` qilamiz. Train'da ko'rilmagan yangi kategoriyalar uchraса, ularni maxsus `"__UNK__"` qiymati bilan belgilaymiz (data leakage'siz yechim).

In [ ]:
ID_COLS = ["RefId"]
categorical_cols = [c for c in df.columns
                     if df[c].dtype == "object" and c not in ID_COLS]
numeric_cols = [c for c in df.columns
                 if c not in categorical_cols + ID_COLS + [TARGET, DATE_COL]]

print("Kategorial belgilar:", categorical_cols)
print("Sonli belgilar:", numeric_cols)

In [ ]:
encoders = {}

for col in categorical_cols:
    train_df[col] = train_df[col].astype(str).fillna("__NA__")
    valid_df[col] = valid_df[col].astype(str).fillna("__NA__")
    test_df[col]  = test_df[col].astype(str).fillna("__NA__")

    le = LabelEncoder()
    le.fit(train_df[col])                      # faqat train'ga fit qilinadi
    known = set(le.classes_)

    # train'da ko'rilmagan qiymatlarni "__UNK__" bilan almashtiramiz
    le_classes = list(le.classes_) + ["__UNK__"]
    le.classes_ = np.array(le_classes)

    def safe_transform(series, encoder, known_set):
        return series.apply(lambda v: v if v in known_set else "__UNK__")

    train_df[col] = le.transform(train_df[col])
    valid_df[col] = le.transform(safe_transform(valid_df[col], le, known))
    test_df[col]  = le.transform(safe_transform(test_df[col], le, known))

    encoders[col] = le

print("Kodlash yakunlandi.")

## 4-vazifa. sklearn modellarini o'qitish (LogisticRegression, GaussianNB, KNN)

O'qitishdan oldin belgilarni `StandardScaler` bilan normallashtiramiz (faqat train'ga `fit`).

In [ ]:
feature_cols = numeric_cols + categorical_cols

X_train = train_df[feature_cols].fillna(0).values
y_train = train_df[TARGET].values
X_valid = valid_df[feature_cols].fillna(0).values
y_valid = valid_df[TARGET].values
X_test  = test_df[feature_cols].fillna(0).values
y_test  = test_df[TARGET].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s  = scaler.transform(X_test)

In [ ]:
def gini_score(y_true, y_scores):
    return abs(2 * roc_auc_score(y_true, y_scores) - 1)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "GaussianNB": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=25),
}

results_step4 = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    p_valid = model.predict_proba(X_valid_s)[:, 1]
    g = gini_score(y_valid, p_valid)
    results_step4[name] = g
    print(f"{name}: Gini(valid) = {g:.4f}")

best_name_step4 = max(results_step4, key=results_step4.get)
print("\nEng yaxshi model:", best_name_step4)

**Talqin:** odatda `LogisticRegression` bu ma'lumot to'plamida eng yaxshi (yoki eng yaqin) natijani beradi, chunki belgilar orasidagi bog'liqlik nisbatan chiziqli va monoton, `GaussianNB` esa belgilarning shartli mustaqilligi taxminini buzganligi sababli chekланган, `KNN` esa yuqori o'lchovlilik (ko'p kategorial ustunlar one-hot/label-encode qilinganda) tufayli "o'lchamlar la'nati"dan aziyat chekishi mumkin. Aniq javob sizning natijangizga (Gini qiymatlariga) bog'liq — yuqoridagi chiqishni tahlil qiling.

## 5-vazifa. Gini score'ni o'zingiz implementatsiya qilish

`Gini = |2 * ROC_AUC - 1|`. ROC AUC'ni rank-statistikasi orqali $O(N\log N)$ vaqtda hisoblaymiz (Mann-Whitney U ehtimoliga ekvivalent formula).

In [ ]:
def my_roc_auc(y_true, y_scores):
    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores, dtype=float)

    order = np.argsort(y_scores)
    ranks = np.empty(len(y_scores))
    ranks[order] = np.arange(1, len(y_scores) + 1)

    # bir xil (tie) qiymatlarga o'rtacha rank beramiz
    sorted_scores = y_scores[order]
    sorted_ranks = ranks[order].astype(float)
    i = 0
    while i < len(sorted_scores):
        j = i
        while j < len(sorted_scores) and sorted_scores[j] == sorted_scores[i]:
            j += 1
        if j - i > 1:
            sorted_ranks[i:j] = sorted_ranks[i:j].mean()
        i = j
    ranks[order] = sorted_ranks

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    sum_ranks_pos = ranks[y_true == 1].sum()

    auc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return auc


def my_gini(y_true, y_scores):
    return abs(2 * my_roc_auc(y_true, y_scores) - 1)

In [ ]:
# Tekshirish: my_gini sklearn asosidagi gini_score'ga teng bo'lishi kerak
for name, model in models.items():
    p_valid = model.predict_proba(X_valid_s)[:, 1]
    g_sklearn = gini_score(y_valid, p_valid)
    g_mine = my_gini(y_valid, p_valid)
    print(f"{name}: sklearn Gini={g_sklearn:.5f}  |  mening Gini'm={g_mine:.5f}  |  farq={abs(g_sklearn-g_mine):.2e}")

## 6-vazifa. LogisticRegression, KNN, NaiveBayes'ni noldan yozish

Har bir model `fit`, `predict_proba`, `predict` (0.5 chegara bilan) metodlariga ega klass ko'rinishida.

### 6.1. Logistic Regression (qo'lda SGD bilan)

Loss gradienti (2-darslikdagi 2.3-bo'limda chiqarilgan): $\nabla_w = X^T(\sigma(Xw+b) - y)/n + \lambda\,\text{sign}(w)$ (L1 bilan), $\nabla_b = \text{mean}(\sigma(Xw+b)-y)$.

In [ ]:
class MyLogisticRegression:
    def __init__(self, lr=0.1, epochs=50, l1=0.0, batch_size=256, random_state=42):
        self.lr = lr
        self.epochs = epochs
        self.l1 = l1
        self.batch_size = batch_size
        self.random_state = random_state

    def _sigmoid(self, z):
        z = np.clip(z, -30, 30)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        y = np.asarray(y, dtype=float)
        for epoch in range(self.epochs):
            idx = rng.permutation(n)
            for start in range(0, n, self.batch_size):
                batch_idx = idx[start:start + self.batch_size]
                xb, yb = X[batch_idx], y[batch_idx]
                z = xb @ self.w + self.b
                p = self._sigmoid(z)
                grad_w = xb.T @ (p - yb) / len(batch_idx) + self.l1 * np.sign(self.w)
                grad_b = np.mean(p - yb)
                self.w -= self.lr * grad_w
                self.b -= self.lr * grad_b
        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

### 6.2. Gaussian Naive Bayes (qo'lda)

In [ ]:
class MyGaussianNB:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors, self.mean, self.var = {}, {}, {}
        for c in self.classes:
            Xc = X[y == c]
            self.priors[c] = Xc.shape[0] / X.shape[0]
            self.mean[c] = Xc.mean(axis=0)
            self.var[c] = Xc.var(axis=0) + 1e-9   # nolga bo'linishning oldini olish
        return self

    def _log_gauss(self, X, mean, var):
        return -0.5 * np.log(2 * np.pi * var) - ((X - mean) ** 2) / (2 * var)

    def predict_proba(self, X):
        log_probs = []
        for c in self.classes:
            lp = np.log(self.priors[c]) + self._log_gauss(X, self.mean[c], self.var[c]).sum(axis=1)
            log_probs.append(lp)
        log_probs = np.vstack(log_probs).T
        log_probs -= log_probs.max(axis=1, keepdims=True)   # sonli barqarorlik uchun
        probs = np.exp(log_probs)
        probs /= probs.sum(axis=1, keepdims=True)
        return probs[:, 1]

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

### 6.3. KNN (qo'lda)

In [ ]:
class MyKNN:
    def __init__(self, k=25):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = np.asarray(y)
        return self

    def predict_proba(self, X):
        probs = np.zeros(X.shape[0])
        for i in range(X.shape[0]):
            dists = np.sqrt(((self.X_train - X[i]) ** 2).sum(axis=1))
            k = min(self.k, len(dists))
            nn_idx = np.argpartition(dists, k - 1)[:k]
            probs[i] = self.y_train[nn_idx].mean()
        return probs

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

**Diqqat:** `MyKNN.predict_proba` katta ma'lumotlarda sekin ishlaydi ($O(N_{test}\cdot N_{train}\cdot D)$), shuning uchun sinov uchun validation to'plamining kichik qismidan (`sample`) foydalanish tavsiya etiladi.

In [ ]:
myLR = MyLogisticRegression(lr=0.5, epochs=100, l1=0.0).fit(X_train_s, y_train)
myNB = MyGaussianNB().fit(X_train_s, y_train)

sample_idx = np.random.RandomState(0).choice(len(X_valid_s), size=min(2000, len(X_valid_s)), replace=False)
myKNN = MyKNN(k=25).fit(X_train_s, y_train)

p_myLR = myLR.predict_proba(X_valid_s)
p_myNB = myNB.predict_proba(X_valid_s)
p_myKNN_sample = myKNN.predict_proba(X_valid_s[sample_idx])

print("Mening LR'im   Gini(valid):", my_gini(y_valid, p_myLR), " | sklearn LR Gini:", results_step4["LogisticRegression"])
print("Mening NB'im   Gini(valid):", my_gini(y_valid, p_myNB), " | sklearn NB Gini:", results_step4["GaussianNB"])
print("Mening KNN'im  Gini(sample):", my_gini(y_valid[sample_idx], p_myKNN_sample),
      " | sklearn KNN Gini(sample):", my_gini(y_valid[sample_idx], models['KNN'].predict_proba(X_valid_s[sample_idx])[:, 1]))

4-vazifadagi natijalarni taxminan takrorlay olishimiz kerak — kichik farqlar normal (sklearn LogisticRegression standart holatda L2 regularizatsiya va L-BFGS optimizatorini ishlatadi, biznikida esa sodda SGD).

## 7-vazifa. Nochiziqli belgilar yaratish

- **Nisbatlar:** ikkita mos sonli belgining bo'linmasi (masalan, `VehOdo / VehicleAge`, nol bo'linishdan saqlanish bilan);
- **Guruh statistikasi:** har bir kategorial ustunning shu guruh bo'yicha o'rtacha `WarrantyCost` (yoki boshqa uzluksiz belgi) bilan kodlanishi — **faqat train**dan hisoblanadi, so'ng valid/test'ga map qilinadi (data leakage'siz).

In [ ]:
def add_features(train, valid, test, numeric_cols, categorical_cols):
    train, valid, test = train.copy(), valid.copy(), test.copy()

    # --- nisbat (ratio) belgi: mavjud bo'lgan ikkita sonli ustundan biror mazmunli nisbat ---
    if "VehOdo" in numeric_cols and "VehicleAge" in numeric_cols:
        for d in (train, valid, test):
            d["ratio_odo_age"] = d["VehOdo"] / (d["VehicleAge"].replace(0, np.nan) + 1)
            d["ratio_odo_age"] = d["ratio_odo_age"].fillna(d["ratio_odo_age"].median() if d is train else train["ratio_odo_age"].median())

    # --- groupby target-encoding uslubidagi belgi: kategoriya -> shu guruhdagi WarrantyCost o'rtachasi ---
    if "WarrantyCost" in numeric_cols:
        for col in categorical_cols:
            group_mean = train.groupby(col)["WarrantyCost"].mean()          # faqat train'dan hisoblanadi
            global_mean = train["WarrantyCost"].mean()
            new_col = f"{col}_warranty_mean"
            train[new_col] = train[col].map(group_mean).fillna(global_mean)
            valid[new_col] = valid[col].map(group_mean).fillna(global_mean)  # train'dagi statistikadan foydalaniladi
            test[new_col]  = test[col].map(group_mean).fillna(global_mean)

    return train, valid, test

train_fe, valid_fe, test_fe = add_features(train_df, valid_df, test_df, numeric_cols, categorical_cols)

new_feature_cols = [c for c in train_fe.columns
                     if c not in ID_COLS + [TARGET, DATE_COL] and train_fe[c].dtype != "object"]
print("Yangi belgilar bilan jami:", len(new_feature_cols), "ta belgi")

In [ ]:
X_train2 = train_fe[new_feature_cols].fillna(0).values
X_valid2 = valid_fe[new_feature_cols].fillna(0).values
X_test2  = test_fe[new_feature_cols].fillna(0).values

scaler2 = StandardScaler()
X_train2_s = scaler2.fit_transform(X_train2)
X_valid2_s = scaler2.transform(X_valid2)
X_test2_s  = scaler2.transform(X_test2)

results_step7 = {}
for name, model_cls in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
                          ("GaussianNB", GaussianNB()),
                          ("KNN", KNeighborsClassifier(n_neighbors=25))]:
    model_cls.fit(X_train2_s, y_train)
    p = model_cls.predict_proba(X_valid2_s)[:, 1]
    g = gini_score(y_valid, p)
    results_step7[name] = g
    print(f"{name}: Gini(valid) = {g:.4f}  (oldingi: {results_step4[name]:.4f}, o'sish: {g - results_step4[name]:+.4f})")

## 8-vazifa. Eng yaxshi belgilarni aniqlash (koeffitsientlar) va L1 orqali tanlash

LogisticRegression koeffitsientlarining mutlaq qiymati — belgining (standartlashtirilgandan keyingi) nisbiy ta'sirini ko'rsatadi.

In [ ]:
lr_full = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(X_train2_s, y_train)
coef_importance = pd.Series(np.abs(lr_full.coef_[0]), index=new_feature_cols).sort_values(ascending=False)
print("Eng muhim 15 ta belgi (koeffitsient bo'yicha):")
print(coef_importance.head(15))

In [ ]:
# --- (a) Qo'lda tanlash: eng past ahamiyatga ega yarmini olib tashlaymiz ---
keep_manual = coef_importance.index[: len(coef_importance) // 2].tolist()
idx_manual = [new_feature_cols.index(c) for c in keep_manual]

lr_manual = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_manual.fit(X_train2_s[:, idx_manual], y_train)
gini_manual = gini_score(y_valid, lr_manual.predict_proba(X_valid2_s[:, idx_manual])[:, 1])

# --- (b) L1 regularizatsiya orqali avtomatik tanlash ---
lr_l1 = LogisticRegression(penalty="l1", solver="liblinear", C=0.5, max_iter=1000, random_state=RANDOM_STATE)
lr_l1.fit(X_train2_s, y_train)
n_nonzero = np.sum(lr_l1.coef_[0] != 0)
gini_l1 = gini_score(y_valid, lr_l1.predict_proba(X_valid2_s)[:, 1])

print(f"Qo'lda tanlash ({len(keep_manual)} belgi): Gini(valid) = {gini_manual:.4f}")
print(f"L1 regularizatsiya  ({n_nonzero} belgi qoldi): Gini(valid) = {gini_l1:.4f}")
print(f"To'liq (barcha {len(new_feature_cols)} belgi): Gini(valid) = {results_step7['LogisticRegression']:.4f}")

**Xulosa uchun savol:** yuqoridagi uchta Gini qiymatini solishtiring. Odatda L1 regularizatsiya qo'lda tanlashdan ko'ra barqarorroq natija beradi, chunki u belgilarning **birgalikdagi** hissasini hisobga oladi (bitta belgining alohida koeffitsienti kichik bo'lsa ham, u boshqa belgilar bilan birga muhim bo'lishi mumkin), qo'lda tanlash esa buni e'tiborga olmaydi.

## 9-vazifa. Eng yaxshi modelning giperparametrlarini sozlash

Eng yaxshi model sifatida (odatda) `LogisticRegression` + to'liq belgi to'plami (yoki L1 orqali tanlangan belgilar) tanlanadi. `C` (regularizatsiya kuchi, teskari) va `penalty` bo'yicha oddiy grid-search o'tkazamiz.

In [ ]:
best_gini_so_far = -1
best_params = None
best_model_obj = None

for C in [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]:
    for penalty in ["l2", "l1"]:
        solver = "liblinear"  # l1 va l2'ni ham qo'llab-quvvatlaydi
        model = LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=1000, random_state=RANDOM_STATE)
        model.fit(X_train2_s, y_train)
        g = gini_score(y_valid, model.predict_proba(X_valid2_s)[:, 1])
        if g > best_gini_so_far:
            best_gini_so_far = g
            best_params = {"C": C, "penalty": penalty}
            best_model_obj = model

print("Eng yaxshi giperparametrlar:", best_params)
print("Eng yaxshi Gini(valid):", best_gini_so_far)

**Kuzatish:** `C` (regularizatsiya kuchini boshqaruvchi parametr) odatda eng katta ta'sirga ega — juda kichik `C` modelni "underfit" qiladi (juda kuchli jazolash), juda katta `C` esa regularizatsiyani deyarli o'chirib, overfitting xavfini oshiradi. `penalty` turi (L1 vs L2) odatda ikkinchi darajali ta'sirga ega, lekin L1 talqin qilinuvchanlikni (kam belgi) qo'shimcha beradi.

## 10-vazifa. Train / Valid / Test bo'yicha Gini solishtiruvi (overfitting tekshiruvi)

In [ ]:
final_model = best_model_obj

gini_train = gini_score(y_train, final_model.predict_proba(X_train2_s)[:, 1])
gini_valid = gini_score(y_valid, final_model.predict_proba(X_valid2_s)[:, 1])
gini_test  = gini_score(y_test,  final_model.predict_proba(X_test2_s)[:, 1])

print(f"Gini (train) = {gini_train:.4f}")
print(f"Gini (valid) = {gini_valid:.4f}")
print(f"Gini (test)  = {gini_test:.4f}")
print(f"\nValid -> Test pasayish: {gini_valid - gini_test:+.4f}")

**Talqin:** agar `train` Gini `valid`/`test` Gini'dan sezilarli darajada yuqori bo'lsa (masalan, 0.05+ farq) — bu **overfitting** belgisi: model train ma'lumotidagi shovqinni "yodlab olgan". Agar uchala qiymat bir-biriga yaqin bo'lsa (masalan, farq 0.01–0.02 atrofida) — model yaxshi umumlashtiryapti (generalizatsiya qilyapti) va overfit emas. Vaqt bo'yicha bo'lingan `test` to'plamida bir oz pasayish odatiy holat (chunki test — kelajakdagi, modelga "notanish" davr), lekin bu pasayish keskin bo'lmasligi kerak.

## 11-vazifa. Recall, Precision, F1 va AUC PR'ni o'zingiz implementatsiya qilish

In [ ]:
def my_precision_recall_f1(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


def my_fbeta(precision, recall, beta):
    num = (1 + beta ** 2) * precision * recall
    den = (beta ** 2) * precision + recall
    return num / den if den > 0 else 0.0


def my_auc_pr(y_true, y_scores):
    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)
    order = np.argsort(-y_scores)          # ehtimol bo'yicha kamayish tartibida
    y_sorted = y_true[order]
    tp_cum = np.cumsum(y_sorted)
    fp_cum = np.cumsum(1 - y_sorted)
    n_pos = y_true.sum()
    precisions = tp_cum / (tp_cum + fp_cum)
    recalls = tp_cum / n_pos
    recalls = np.concatenate(([0], recalls))
    precisions = np.concatenate(([1], precisions))
    return np.sum((recalls[1:] - recalls[:-1]) * precisions[1:])   # trapezoidal integratsiya

In [ ]:
print("=== Test to'plamida solishtiruv (AUC PR) ===\n")

test_scores_dict = {}

# sklearn modellari (7-vazifadagi feature-set bilan qayta o'qitilgan)
for name, m in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
                 ("GaussianNB", GaussianNB()),
                 ("KNN", KNeighborsClassifier(n_neighbors=25))]:
    m.fit(X_train2_s, y_train)
    test_scores_dict[name] = m.predict_proba(X_test2_s)[:, 1]

for name, scores in test_scores_dict.items():
    preds = (scores >= 0.5).astype(int)
    p, r, f1 = my_precision_recall_f1(y_test, preds)
    aucpr_mine = my_auc_pr(y_test, scores)
    aucpr_sklearn = average_precision_score(y_test, scores)
    print(f"{name:20s} | Precision={p:.4f}  Recall={r:.4f}  F1={f1:.4f}  "
          f"AUC_PR(mening)={aucpr_mine:.4f}  AUC_PR(sklearn AP)={aucpr_sklearn:.4f}")

## 12-vazifa. "Lemon" (nuqsonli) mashinalarni aniqlash uchun qaysi metrika afzal?

**Kontekst:** `IsBadBuy=1` — auktsionda sotib olingan mashina "lemon" (yashirin nuqsonli, keyinchalik ko'p xarajat keltiradigan) bo'lib chiqishi. Bu — avtomobil dilerlari uchun **moliyaviy zarar** bilan bog'liq masala.

**Tahlil:**
- **False Negative (FN)** — model "yaxshi" deb bashorat qilgan, lekin aslida "lemon" bo'lgan mashina. Diler bunday mashinani sotib oladi va katta zarar ko'radi (ta'mirlash xarajatlari, mijozlar noroziligi, obro' yo'qotish).
- **False Positive (FP)** — model "lemon" deb bashorat qilgan, lekin aslida yaxshi mashina. Bunda diler shunchaki yaxshi imkoniyatni o'tkazib yuboradi (upushchennaya foyda) — bu ham yomon, lekin FN'dan ancha arzonroq.

Demak, FN narxi FP narxidan sezilarli darajada yuqori — bu vaziyat **Precision'dan ko'ra Recall'ga ko'proq e'tibor** berishni talab qiladi (README'dagi "Medicine" va "Fraud detection" misollariga o'xshash holat): biz iloji boricha ko'proq haqiqiy "lemon" mashinalarni "ushlab qolishimiz" kerak, hattoki bunda ba'zi yaxshi mashinalarni ham noto'g'ri belgilab qo'ysak ham.

**Xulosa:** eng mos qattiq yorliq (hard label) metrikasi — **Recall** (yoki **Fbeta score, $\beta>1$ bilan**, masalan $F_2$), chunki u Recall'ga ko'proq og'irlik beradi va shu bilan birga Precision'ni butunlay e'tiborsiz qoldirmaydi (agar Precision juda past bo'lsa, diler foydali bo'lishi mumkin bo'lgan juda ko'p mashinani rad etib, sotib olish hajmini keskin kamaytirib yuboradi — bu ham amaliy jihatdan yomon).

In [ ]:
beta = 2  # Recall'ga Precision'dan ko'ra ko'proq og'irlik beramiz

for name, scores in test_scores_dict.items():
    preds = (scores >= 0.5).astype(int)
    p, r, f1 = my_precision_recall_f1(y_test, preds)
    f2 = my_fbeta(p, r, beta=beta)
    print(f"{name:20s} | Precision={p:.4f}  Recall={r:.4f}  F1={f1:.4f}  F2(beta=2)={f2:.4f}")